## MAI-Transcribe-1.5 (Foundry integration patterns)

This notebook follows Foundry-oriented integration patterns for MAI-Transcribe-1.5:
- Speech SDK real-time transcription (microphone)
- `POST /speechtotext/transcriptions:transcribe?api-version=2025-10-15` (fast + LLM speech)
- batch transcription submission via REST


## 1. Setup

### Environment variables

| Variable | Required | Secret | Purpose |
|---|---|---|---|
| `MAI_TRANSCRIBE_15_ENDPOINT` | Optional | No | Foundry/Speech endpoint (for example `https://<resource>.cognitiveservices.azure.com/`). |
| `MAI_TRANSCRIBE_15_KEY` | Yes | **Yes** | Key used by Speech SDK and REST (`Ocp-Apim-Subscription-Key`). |
| `MAI_TRANSCRIBE_15_REGION` | Optional | No | Region for batch REST calls (for example `swedencentral`). |
| `TRANSCRIBE_LOCAL_AUDIO_DIR` | Optional | No | Folder containing local WAV/MP3/FLAC files; defaults to `media/mai-transcribe-1-5`. |

Do not commit `.env` or `deployment.env` files with secrets.


In [ ]:
# %pip install -q requests python-dotenv azure-cognitiveservices-speech

In [ ]:
import os
import json
from copy import deepcopy
from pathlib import Path
import requests
import azure.cognitiveservices.speech as speechsdk
from dotenv import load_dotenv

ENV_PATH = 'deployment.env' if os.path.exists('deployment.env') else os.path.join('..', 'deployment.env')
load_dotenv(ENV_PATH, override=True)

SPEECH_ENDPOINT = (
    os.getenv('MAI_TRANSCRIBE_15_ENDPOINT')
    or os.getenv('AZURE_SPEECH_ENDPOINT')
    or 'https://voiceliveapipoc.cognitiveservices.azure.com/'
).rstrip('/')
SPEECH_KEY = os.getenv('MAI_TRANSCRIBE_15_KEY')
SERVICE_REGION = os.getenv('MAI_TRANSCRIBE_15_REGION') or os.getenv('AZURE_SPEECH_REGION') or 'swedencentral'
TRANSCRIBE_URL = f"{SPEECH_ENDPOINT}/speechtotext/transcriptions:transcribe?api-version=2025-10-15"
BATCH_SUBMIT_URL = f"https://{SERVICE_REGION}.api.cognitive.microsoft.com/speechtotext/transcriptions:submit?api-version=2024-11-15"
assert SPEECH_KEY, 'Set MAI_TRANSCRIBE_15_KEY in deployment.env'

audio_dir_env = os.getenv('TRANSCRIBE_LOCAL_AUDIO_DIR')
LOCAL_AUDIO_DIR = Path(audio_dir_env) if audio_dir_env else Path('media') / 'mai-transcribe-1-5'
assert LOCAL_AUDIO_DIR.exists(), f'Audio folder not found: {LOCAL_AUDIO_DIR}'
candidates = sorted(LOCAL_AUDIO_DIR.glob('*.wav')) + sorted(LOCAL_AUDIO_DIR.glob('*.mp3')) + sorted(LOCAL_AUDIO_DIR.glob('*.flac'))
non_empty = [p for p in candidates if p.stat().st_size > 0]
assert non_empty, f'No non-empty WAV/MP3/FLAC files found in: {LOCAL_AUDIO_DIR}'
AUDIO_FILE = str(max(non_empty, key=lambda p: p.stat().st_size))
print('Endpoint:', SPEECH_ENDPOINT)
print('Batch region:', SERVICE_REGION)
print('Audio file:', AUDIO_FILE)

def _mime_for(path: Path) -> str:
    s = path.suffix.lower()
    if s == '.wav':
        return 'audio/wav'
    if s == '.mp3':
        return 'audio/mpeg'
    if s == '.flac':
        return 'audio/flac'
    return 'application/octet-stream'

def transcribe_with_definition(audio_path: str, definition: dict) -> dict:
    p = Path(audio_path)
    with p.open('rb') as f:
        files = {
            'audio': (p.name, f, _mime_for(p)),
            'definition': (None, json.dumps(definition), 'application/json'),
        }
        resp = requests.post(
            TRANSCRIBE_URL,
            headers={'Ocp-Apim-Subscription-Key': SPEECH_KEY},
            files=files,
            timeout=300,
        )
    if resp.ok:
        return resp.json()

    # Compatibility fallback for backends that reject enhancedMode/model
    txt = (resp.text or '').lower()
    if resp.status_code == 400 and 'enhanced mode with model' in txt:
        fb = deepcopy(definition)
        fb.get('enhancedMode', {}).pop('model', None)
        return transcribe_with_definition(audio_path, fb)
    if resp.status_code == 400 and 'enhanced mode is currently not supported' in txt:
        fb = deepcopy(definition)
        fb.pop('enhancedMode', None)
        return transcribe_with_definition(audio_path, fb)

    raise requests.HTTPError(f'Transcription failed with {resp.status_code}: {resp.text}', response=resp)


Illustrative sample audio used in this recipe:

- [Sample audio (`sample-en.mp3`)](media/mai-transcribe-1-5/sample-en.mp3)

Transcription results are illustrative and may vary by model updates, locale detection, and audio quality.


## 2. Real-time transcription (Speech SDK microphone)

In [ ]:
realtime_speech_config = speechsdk.SpeechConfig(subscription=SPEECH_KEY, endpoint=SPEECH_ENDPOINT)
audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
speech_recognizer = speechsdk.SpeechRecognizer(
    speech_config=realtime_speech_config,
    audio_config=audio_config,
)

def recognized_handler(evt):
    print(f"Recognized: {evt.result.text}")

def recognizing_handler(evt):
    print(f"Recognizing: {evt.result.text}")

RUN_REALTIME = False
if RUN_REALTIME:
    speech_recognizer.recognized.connect(recognized_handler)
    speech_recognizer.recognizing.connect(recognizing_handler)
    speech_recognizer.start_continuous_recognition()
    print('Say something...')
    input('Press Enter to stop...')
    speech_recognizer.stop_continuous_recognition()
else:
    print('Set RUN_REALTIME=True to run microphone real-time transcription.')


Example console output:

```json
Recognizing: Hello, welcome to Azure AI Foundry...
Recognized: Hello, welcome to Azure AI Foundry.
```


## 3. Fast transcription (multipart REST with diarization)

In [ ]:
fast_definition = {
    'locales': ['en-US'],
    'diarization': {
        'maxSpeakers': 2,
        'enabled': True,
    },
}
fast_result = transcribe_with_definition(AUDIO_FILE, fast_definition)
print(json.dumps(fast_result, indent=2)[:2000])


Example response shape:

```json
{
  "id": "<redacted>",
  "status": "succeeded",
  "combinedPhrases": [
    {"text": "..."}
  ],
  "phrases": [
    {"speaker": 0, "offsetMilliseconds": 0, "text": "..."}
  ]
}
```


## 4. LLM speech transcription (enhanced mode)

Uses `enhancedMode.task = transcribe` as shown in the Foundry sample.


In [ ]:
llm_definition = {
    'enhancedMode': {
        'enabled': True,
        'task': 'transcribe',
    },
}
llm_result = transcribe_with_definition(AUDIO_FILE, llm_definition)
print(json.dumps(llm_result, indent=2)[:2000])


Example response shape:

```json
{
  "id": "<redacted>",
  "status": "succeeded",
  "combinedPhrases": [
    {"text": "..."}
  ],
  "phrases": [
    {"speaker": 0, "offsetMilliseconds": 0, "text": "..."}
  ]
}
```


## 5. Entity biasing example (phrase list)

This is the entity biasing pattern: pass important names/terms via `phraseList.phrases` to improve recognition.

In [ ]:
phrase_list_definition = {
    'phraseList': {
        'phrases': ['MAI', 'Microsoft Build', 'Foundry', 'Copilot']
    },
    'enhancedMode': {
        'enabled': True,
        'task': 'transcribe',
    },
}
phrase_list_result = transcribe_with_definition(AUDIO_FILE, phrase_list_definition)
print(json.dumps(phrase_list_result, indent=2)[:2000])


Example response shape:

```json
{
  "id": "<redacted>",
  "status": "succeeded",
  "combinedPhrases": [
    {"text": "..."}
  ],
  "phrases": [
    {"speaker": 0, "offsetMilliseconds": 0, "text": "..."}
  ]
}
```


## 6. Batch transcription (REST submit pattern)

```python
batch_payload = {
  "contentUrls": [
    "https://crbn.us/hello.wav",
    "https://crbn.us/whatstheweatherlike.wav"
  ],
  "locale": "en-US",
  "displayName": "MAI-Transcribe-1.5 Batch Sample",
  "model": None,
  "properties": {
    "wordLevelTimestampsEnabled": True,
    "languageIdentification": {
      "candidateLocales": ["en-US", "de-DE", "es-ES"],
      "mode": "Continuous"
    },
    "timeToLiveHours": 48
  }
}

resp = requests.post(
    BATCH_SUBMIT_URL,
    headers={
        'Ocp-Apim-Subscription-Key': SPEECH_KEY,
        'Content-Type': 'application/json',
    },
    data=json.dumps(batch_payload),
    timeout=300,
)
resp.raise_for_status()
print(resp.json())
```


## 7. Notes and troubleshooting

| Error | Resolution |
|---|---|
| `Enhanced mode ... not supported` | Backend limitation; helper auto-falls back by removing unsupported `enhancedMode` fields. |
| `InvalidLocale` | Add/adjust locale in `definition` for fast/batch patterns. |
| `EmptyAudioFile` | Use a non-empty file; notebook auto-picks largest non-empty local audio file. |
| `401/403` | Verify `MAI_TRANSCRIBE_15_KEY` and `MAI_TRANSCRIBE_15_ENDPOINT` in `deployment.env`. |

Model-card notes: MAI-Transcribe-1.5 supports expanded language coverage, faster long-form inference, and keyword/entity biasing.
